<a href="https://colab.research.google.com/github/2303a51899/RL/blob/main/RL_LAB_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
gamma = 0.9
theta = 1e-6

states = ['S0', 'S1', 'G']
non_terminal_states = ['S0', 'S1']
actions = ['Up', 'Down', 'Left', 'Right']

step_reward = -1.0
wall_reward = -1.0
goal_reward = 100.0

transitions = {
    'S0': {
        'Up': ('S0', wall_reward),
        'Down': ('S1', step_reward),
        'Left': ('S0', wall_reward),
        'Right': ('S0', wall_reward),
    },
    'S1': {
        'Up': ('S0', step_reward),
        'Down': ('S1', wall_reward),
        'Left': ('S1', wall_reward),
        'Right': ('G', goal_reward),
    },
}

print('=== 1. MDP DEFINITION ===')
print(f'States              : {states}')
print(f'Non-Terminal States : {non_terminal_states}')
print(f'Actions             : {actions}')
print(
    f'Rewards             : Step={step_reward}, Wall={wall_reward},'
    f' Goal={goal_reward}'
)
print(f'Discount Factor     : {gamma}\n')

print('=== TRANSITION TABLE ===')
for s in non_terminal_states:
  for a in actions:
    next_s, r = transitions[s][a]
    print(f'State: {s:<3} | Action: {a:<6} -> Next State: {next_s:<3} | Reward: {r}')
print()

policy = {'S0': 'Up', 'S1': 'Up'}
V = {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

print('=== 2. INITIAL POLICY & STATE VALUES ===')
print(f'Initial Policy: {policy}')
print(f'Initial Values: {V}\n')

pi_iter = 0
while True:
  pi_iter += 1
  print(f'--- Policy Iteration {pi_iter} ---')
  print(f'Current Policy: {policy}')

  eval_iter = 0
  while True:
    eval_iter += 1
    delta = 0.0
    V_new = V.copy()

    for s in non_terminal_states:
      a = policy[s]
      next_s, r = transitions[s][a]
      V_new[s] = r + gamma * V[next_s]
      delta = max(delta, abs(V_new[s] - V[s]))

    V = V_new
    print(
        f'  Eval Iter {eval_iter:02d} | V(S0) = {V["S0"]:.4f} | V(S1) ='
        f' {V["S1"]:.4f} | Delta = {delta:.6f}'
    )
    if delta < theta:
      break

  Q = {s: {} for s in non_terminal_states}
  new_policy = {}
  for s in non_terminal_states:
    for a in actions:
      next_s, r = transitions[s][a]
      Q[s][a] = r + gamma * V[next_s]
    new_policy[s] = max(Q[s], key=Q[s].get)

  print(f'Improved Policy: {new_policy}\n')

  if new_policy == policy:
    break
  policy = new_policy

print('=== FINAL OPTIMAL STATE VALUES V*(s) ===')
for s in states:
  print(f'V*({s}) = {V[s]:.4f}')
print()

print('=== FINAL ACTION-VALUE TABLE Q*(s, a) ===')
for s in non_terminal_states:
  for a in actions:
    print(f'Q*({s}, {a:<5}) = {Q[s][a]:.4f}')
print()

print('=== OPTIMAL POLICY & EXPECTED RESULTS ===')
for s in non_terminal_states:
  print(f'State: {s:<3} | Optimal Value: {V[s]:>7.2f} | Optimal Action: {policy[s]}')
print(f'State: G   | Optimal Value: {V["G"]:>7.2f} | Optimal Action: Terminal\n')

print('=== BELLMAN OPTIMALITY VERIFICATION ===')
for s in non_terminal_states:
  max_q = max(Q[s].values())
  verified = abs(V[s] - max_q) < 1e-5
  print(
      f'{s}: V*({s}) = {V[s]:.4f}, max_a Q*({s}, a) = {max_q:.4f} -> Verified:'
      f' {verified}'
  )

=== 1. MDP DEFINITION ===
States              : ['S0', 'S1', 'G']
Non-Terminal States : ['S0', 'S1']
Actions             : ['Up', 'Down', 'Left', 'Right']
Rewards             : Step=-1.0, Wall=-1.0, Goal=100.0
Discount Factor     : 0.9

=== TRANSITION TABLE ===
State: S0  | Action: Up     -> Next State: S0  | Reward: -1.0
State: S0  | Action: Down   -> Next State: S1  | Reward: -1.0
State: S0  | Action: Left   -> Next State: S0  | Reward: -1.0
State: S0  | Action: Right  -> Next State: S0  | Reward: -1.0
State: S1  | Action: Up     -> Next State: S0  | Reward: -1.0
State: S1  | Action: Down   -> Next State: S1  | Reward: -1.0
State: S1  | Action: Left   -> Next State: S1  | Reward: -1.0
State: S1  | Action: Right  -> Next State: G   | Reward: 100.0

=== 2. INITIAL POLICY & STATE VALUES ===
Initial Policy: {'S0': 'Up', 'S1': 'Up'}
Initial Values: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

--- Policy Iteration 1 ---
Current Policy: {'S0': 'Up', 'S1': 'Up'}
  Eval Iter 01 | V(S0) = -1.0000 | V(S1)